In [1]:
from __future__ import annotations

from pathlib import Path

import h5py
import numpy as np


SEED = 0

OUTPUT_PATH = Path("data/ceql_ablation_3var_linear_rational.h5")
GROUP_NAME = "A1_three_variate_linear_rational"

N_TRAIN = 128
N_TEST_INTERP = 8192
N_TEST_EXTRAP = 8192

N_FEATURES = 3

TRAIN_LOW = -2.0
TRAIN_HIGH = 2.0

EXTRAP_A_LOW = -4.0
EXTRAP_A_HIGH = -2.0
EXTRAP_B_LOW = 2.0
EXTRAP_B_HIGH = 4.0

COEFF_MIN_ABS = 0.5
COEFF_MAX_ABS = 3.0
COEFF_DECIMALS = 2

MAX_EXPR_TRIALS = 10000

BATCH_SIZE = 8192
MAX_SAMPLE_ROUNDS = 200

POLE_EPS = 1e-3
MAX_ABS_Y = 1e2

REQUIRE_DENOMINATOR_SIGN_CHANGE_ON_TRAIN_DOMAIN = True


def sample_signed_coefficients(rng: np.random.Generator, n: int) -> np.ndarray:
    magnitudes = rng.uniform(COEFF_MIN_ABS, COEFF_MAX_ABS, size=n)
    signs = rng.choice(np.array([-1.0, 1.0]), size=n)
    coefficients = signs * magnitudes
    return np.round(coefficients, COEFF_DECIMALS)


def linear_eval(X: np.ndarray, c: np.ndarray) -> np.ndarray:
    x1 = X[:, 0].astype(np.float64)
    x2 = X[:, 1].astype(np.float64)
    x3 = X[:, 2].astype(np.float64)

    return c[0] * x1 + c[1] * x2 + c[2] * x3 + c[3]


def box_corners() -> np.ndarray:
    values = [TRAIN_LOW, TRAIN_HIGH]
    corners = []

    for x1 in values:
        for x2 in values:
            for x3 in values:
                corners.append([x1, x2, x3])

    return np.array(corners, dtype=np.float64)


def denominator_has_zero_set_in_train_domain(c_den: np.ndarray) -> bool:
    den = linear_eval(box_corners(), c_den)
    return float(den.min()) < 0.0 and float(den.max()) > 0.0


def sample_expression_coefficients(rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    for _ in range(MAX_EXPR_TRIALS):
        c_num = sample_signed_coefficients(rng, 4)
        c_den = sample_signed_coefficients(rng, 4)

        if REQUIRE_DENOMINATOR_SIGN_CHANGE_ON_TRAIN_DOMAIN:
            if not denominator_has_zero_set_in_train_domain(c_den):
                continue

        return c_num, c_den

    raise RuntimeError("Could not sample a valid rational expression.")


def eval_rational(
    X: np.ndarray,
    c_num: np.ndarray,
    c_den: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    numerator = linear_eval(X, c_num)
    denominator = linear_eval(X, c_den)
    y = numerator / denominator

    return y, denominator


def sample_train_or_interp_points(rng: np.random.Generator, n: int) -> np.ndarray:
    X = rng.uniform(TRAIN_LOW, TRAIN_HIGH, size=(n, N_FEATURES))
    return X.astype(np.float32)


def sample_extrap_points(rng: np.random.Generator, n: int) -> np.ndarray:
    side = rng.integers(0, 2, size=(n, N_FEATURES))

    left = rng.uniform(EXTRAP_A_LOW, EXTRAP_A_HIGH, size=(n, N_FEATURES))
    right = rng.uniform(EXTRAP_B_LOW, EXTRAP_B_HIGH, size=(n, N_FEATURES))

    X = np.where(side == 0, left, right)
    return X.astype(np.float32)


def make_split(
    rng: np.random.Generator,
    n_required: int,
    split: str,
    c_num: np.ndarray,
    c_den: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    X_parts = []
    y_parts = []
    n_collected = 0

    for _ in range(MAX_SAMPLE_ROUNDS):
        if split == "train" or split == "test_interp":
            X = sample_train_or_interp_points(rng, BATCH_SIZE)
        elif split == "test_extrap":
            X = sample_extrap_points(rng, BATCH_SIZE)
        else:
            raise ValueError(split)

        y, den = eval_rational(X, c_num, c_den)

        mask = np.isfinite(y)
        mask &= np.isfinite(den)
        mask &= np.abs(den) > POLE_EPS
        mask &= np.abs(y) <= MAX_ABS_Y

        X_valid = X[mask]
        y_valid = y[mask]

        X_parts.append(X_valid)
        y_parts.append(y_valid)

        n_collected += X_valid.shape[0]

        if n_collected >= n_required:
            break

    if n_collected < n_required:
        raise RuntimeError(
            f"Could not generate enough valid samples for {split}. "
            f"Collected {n_collected}, required {n_required}."
        )

    X_out = np.concatenate(X_parts, axis=0)[:n_required].astype(np.float32)
    y_out = np.concatenate(y_parts, axis=0)[:n_required].astype(np.float32)

    return X_out, y_out


def coefficient_to_string(value: float) -> str:
    return f"{value:.{COEFF_DECIMALS}f}"


def signed_term_string(value: float, monomial: str | None, is_first: bool) -> str:
    abs_value = abs(float(value))
    coefficient = coefficient_to_string(abs_value)

    if monomial is None:
        term = coefficient
    else:
        term = f"{coefficient}*{monomial}"

    if is_first:
        if value < 0.0:
            return f"-{term}"
        return term

    if value < 0.0:
        return f" - {term}"

    return f" + {term}"


def linear_expression_string(c: np.ndarray) -> str:
    monomials = ["x1", "x2", "x3", None]

    terms = [
        signed_term_string(c[i], monomials[i], is_first=(i == 0))
        for i in range(4)
    ]

    return "".join(terms)


def rational_expression_string(c_num: np.ndarray, c_den: np.ndarray) -> str:
    numerator = linear_expression_string(c_num)
    denominator = linear_expression_string(c_den)

    return f"({numerator})/({denominator})"


def main() -> None:
    rng = np.random.default_rng(SEED)

    c_num, c_den = sample_expression_coefficients(rng)

    X_train, y_train = make_split(rng, N_TRAIN, "train", c_num, c_den)
    X_test_interp, y_test_interp = make_split(rng, N_TEST_INTERP, "test_interp", c_num, c_den)
    X_test_extrap, y_test_extrap = make_split(rng, N_TEST_EXTRAP, "test_extrap", c_num, c_den)

    expr_str = rational_expression_string(c_num, c_den)

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    with h5py.File(OUTPUT_PATH, "w") as f:
        g = f.create_group(GROUP_NAME)

        g.create_dataset("sympy_str", data=expr_str.encode("utf-8"))

        g.attrs["seed"] = SEED
        g.attrs["n_features"] = N_FEATURES
        g.attrs["coeff_min_abs"] = COEFF_MIN_ABS
        g.attrs["coeff_max_abs"] = COEFF_MAX_ABS
        g.attrs["coeff_decimals"] = COEFF_DECIMALS
        g.attrs["train_low"] = TRAIN_LOW
        g.attrs["train_high"] = TRAIN_HIGH
        g.attrs["extrap_a_low"] = EXTRAP_A_LOW
        g.attrs["extrap_a_high"] = EXTRAP_A_HIGH
        g.attrs["extrap_b_low"] = EXTRAP_B_LOW
        g.attrs["extrap_b_high"] = EXTRAP_B_HIGH
        g.attrs["pole_eps"] = POLE_EPS
        g.attrs["max_abs_y"] = MAX_ABS_Y

        g.create_dataset("numerator_coefficients", data=c_num)
        g.create_dataset("denominator_coefficients", data=c_den)

        train = g.create_group("train")
        train.create_dataset("X", data=X_train)
        train.create_dataset("y", data=y_train)

        test_interp = g.create_group("test_interp")
        test_interp.create_dataset("X", data=X_test_interp)
        test_interp.create_dataset("y", data=y_test_interp)

        test_extrap = g.create_group("test_extrap")
        test_extrap.create_dataset("X", data=X_test_extrap)
        test_extrap.create_dataset("y", data=y_test_extrap)

    print(f"Saved dataset to: {OUTPUT_PATH}")
    print(f"Group name: {GROUP_NAME}")
    print()
    print("Numerator coefficients:")
    print(c_num)
    print()
    print("Denominator coefficients:")
    print(c_den)
    print()
    print("Saved SymPy expression string:")
    print(expr_str)


if __name__ == "__main__":
    main()

Saved dataset to: data/ceql_ablation_3var_linear_rational.h5
Group name: A1_three_variate_linear_rational

Numerator coefficients:
[-2.09  1.17  0.6   0.54]

Denominator coefficients:
[-2.02  2.32  1.86 -2.84]

Saved SymPy expression string:
(-2.09*x1 + 1.17*x2 + 0.60*x3 + 0.54)/(-2.02*x1 + 2.32*x2 + 1.86*x3 - 2.84)
